# Datengenerierung für das Data Management Praxisbeispiel
### Kontext: Threat Hunting & Cybersecurity Management

Dieses Notebook generiert zwei synthetische Datensätze für eine interaktive ETL- und Visualisierungsübung (z. B. in KNIME oder direkt in Python):
1. **unternehmens_proxy_logs.csv**: Ein Protokoll von 3.000 Web-Zugriffen aus einem Firmennetzwerk über den Zeitraum einer Woche rückwärtsgerichtet bis zum **03.06.2026, 13:30 Uhr**.
2. **threat_intel_blacklist.csv**: Eine tagesaktuelle Bedrohungs-Blacklist mit bekannten bösartigen Domains zur Fusionierung (Join).

## 1. Imports und Konfiguration

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Seed für reproduzierbare Ergebnisse setzen
np.random.seed(42)
random.seed(42)

num_rows = 5000
end_time = datetime(2026, 6, 3, 13, 30, 0)
start_time = end_time - timedelta(days=7)
print(f"Generierungszeitraum: {start_time} bis {end_time}")

Generierungszeitraum: 2026-05-27 13:30:00 bis 2026-06-03 13:30:00


## 2. Definition der Domains (100 Gutartige & 5 Bösartige)

In [2]:
benign_domains = [
    # Suchmaschinen & Tech-Giganten (10)
    "google.com", "google.de", "bing.com", "duckduckgo.com", "yahoo.com", 
    "apple.com", "microsoft.com", "maps.google.com", "accounts.google.com", "support.microsoft.com",
    
    # Hochschule, Wissenschaft & Bildung (10)
    "hs-mainz.de", "uni-mainz.de", "researchgate.net", "sciencedirect.com", "springer.com", 
    "wiley.com", "ieee.org", "arxiv.org", "coursera.org", "udemy.com",
    
    # Entwicklung, Open Source & Programmierung (15)
    "github.com", "gitlab.com", "bitbucket.org", "stackoverflow.com", "stackexchange.com", 
    "docker.com", "python.org", "npmjs.com", "w3schools.com", "mozilla.org", 
    "pypi.org", "developer.mozilla.org", "github.io", "codepen.io", "jsfiddle.net",
    
    # Kollaboration & Unternehmens-Kommunikation (15)
    "outlook.office365.com", "teams.microsoft.com", "slack.com", "zoom.us", "webex.com", 
    "atlassian.net", "jira.com", "confluence.com", "notion.so", "miro.com", 
    "trello.com", "dropbox.com", "box.com", "sharepoint.com", "office.com",
    
    # Cloud & Enterprise-Software (10)
    "salesforce.com", "sap.com", "hubspot.com", "oracle.com", "ibm.com", 
    "workday.com", "adobe.com", "aws.amazon.com", "portal.azure.com", "cloud.google.com",
    
    # Deutsche Medien & IT-Nachrichten (15)
    "heise.de", "golem.de", "spiegel.de", "zeit.de", "faz.net", 
    "sueddeutsche.de", "welt.de", "tagesschau.de", "zdf.de", "ard.de", 
    "n-tv.de", "focus.de", "handelsblatt.com", "wiwo.de", "computerbase.de",
    
    # Internationale Medien & Wissen (10)
    "wikipedia.org", "wikimedia.org", "bbc.co.uk", "cnn.com", "nytimes.com", 
    "reuters.com", "bloomberg.com", "forbes.com", "theguardian.com", "techcrunch.com",
    
    # Netzwerke, Finanzen, Logistik & CDNs (15)
    "linkedin.com", "xing.com", "paypal.com", "stripe.com", "dhl.de", 
    "fedex.com", "telekom.de", "bahn.de", "fonts.googleapis.com", "gstatic.com", 
    "cdnjs.cloudflare.com", "amazon.de", "ebay.de", "cloudflare.com", "raw.githubusercontent.com"
]

malicious_domains = [
    "malicious-site.ru", 
    "bad-ransomware-c2.net", 
    "phishing-bank-login.updates-secure.com", 
    "download-malware.cc", 
    "test-crypto-miner.org"
]

print(f"Anzahl gutartiger Domains: {len(benign_domains)}")
print(f"Anzahl bösartiger Domains: {len(malicious_domains)}")

Anzahl gutartiger Domains: 100
Anzahl bösartiger Domains: 5


## 3. Generierung der Proxy-Log-Daten
Hier erstellen wir die in num_rows angebene Anzahl von Zeilen. Für die bösartige Domain `bad-ransomware-c2.net` simulieren wir in einigen Fällen einen massiven Datenabfluss (`Bytes_Sent` zwischen 55 und 75 MB), um den Studierenden ein spannendes Analyse-Szenario zu bieten.

In [3]:
# Zeitstempel zufällig im Intervall generieren und sortieren
timestamps = [start_time + timedelta(seconds=random.randint(0, int((end_time - start_time).total_seconds()))) for _ in range(num_rows)]
timestamps.sort()

# 50 Benutzer-IDs und deren statische IPs im Firmennetzwerk erzeugen
users = [f"USR{i:03d}" for i in range(1, 51)]
user_ip_map = {user: f"10.15.2.{i+10}" for i, user in enumerate(users)}

# Gewichtung: Gutartige Domains tauchen viel häufiger auf
domain_choices = benign_domains * 25 + malicious_domains
chosen_domains = [random.choice(domain_choices) for _ in range(num_rows)]

log_data = []
for i in range(num_rows):
    ts = timestamps[i].strftime("%Y-%m-%d %H:%M:%S")
    user = random.choice(users)
    ip = user_ip_map[user]
    domain = chosen_domains[i]
    
    # Standard-Traffic-Größen (in Bytes)
    b_sent = random.randint(500, 50000)
    b_recv = random.randint(2000, 1500000)
    
    # Injektion der Anomalie: Datenabfluss (Data Exfiltration) über Command & Control (C2)
    if domain == "bad-ransomware-c2.net" and random.random() < 0.22:
        print("Anomalie generiert: Datenabfluss über C2 bei 'bad-ransomware-c2.net'.")
        b_sent = random.randint(55000000, 75000000) # ~55 bis 75 MB Upload!
        b_recv = random.randint(10000, 50000)
        
    log_data.append([ts, user, ip, domain, b_sent, b_recv])

# In DataFrame umwandeln
proxy_df = pd.DataFrame(log_data, columns=["Timestamp", "User_ID", "Source_IP", "Requested_Domain", "Bytes_Sent", "Bytes_Received"])

# Als CSV exportieren
import os
os.makedirs("../data", exist_ok=True)
proxy_df.to_csv("../data/unternehmens_proxy_logs.csv", index=False)
print("Datei 'unternehmens_proxy_logs.csv' mit "  + str(num_rows) +  " Einträgen generiert.")


Anomalie generiert: Datenabfluss über C2 bei 'bad-ransomware-c2.net'.
Datei 'unternehmens_proxy_logs.csv' mit 5000 Einträgen generiert.


## 4. Generierung der Threat Intelligence Blacklist

In [4]:
blacklist_data = [
    ["malicious-site.ru", "Malware_Distribution", "High"],
    ["bad-ransomware-c2.net", "Command_and_Control", "Critical"],
    ["phishing-bank-login.updates-secure.com", "Phishing", "Medium"],
    ["download-malware.cc", "Malware_Distribution", "High"],
    ["test-crypto-miner.org", "Cryptomining", "Low"]
]

blacklist_df = pd.DataFrame(blacklist_data, columns=["Domain", "Threat_Type", "Risk_Level"])

# Als CSV exportieren
import os
os.makedirs("../data", exist_ok=True)
blacklist_df.to_csv("../data/threat_intel_blacklist.csv", index=False)
print("Datei 'threat_intel_blacklist.csv' erfolgreich exportiert.")


Datei 'threat_intel_blacklist.csv' erfolgreich exportiert.


## 5. Datenüberprüfung (Quick Check)
Hier validieren wir kurz die erzeugten Strukturen, um sicherzustellen, dass alles bereit für die Lehre ist.

In [5]:
print("--- PROXY LOGS VORSCHAU ---")
display(proxy_df.head())

print("\n--- ANOMALIEN-CHECK (Massiver Daten-Upload) ---")
exfil_cases = proxy_df[proxy_df["Bytes_Sent"] > 10000000]
display(exfil_cases)

--- PROXY LOGS VORSCHAU ---


,Timestamp,User_ID,Source_IP,Requested_Domain,Bytes_Sent,Bytes_Received
0,2026-05-27 13:31:36,USR005,10.15.2.14,portal.azure.com,24555,281589
1,2026-05-27 13:32:57,USR033,10.15.2.42,accounts.google.com,23653,108758
2,2026-05-27 13:35:01,USR028,10.15.2.37,computerbase.de,32973,797019
3,2026-05-27 13:37:05,USR011,10.15.2.20,gitlab.com,24963,239467
4,2026-05-27 13:39:59,USR022,10.15.2.31,telekom.de,45003,525795



--- ANOMALIEN-CHECK (Massiver Daten-Upload) ---


,Timestamp,User_ID,Source_IP,Requested_Domain,Bytes_Sent,Bytes_Received
3653,2026-06-01 17:05:20,USR022,10.15.2.31,bad-ransomware-c2.net,61000797,39841
